In [1]:
import os, warnings 
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

python-dotenv could not parse statement starting at line 8


True

In [2]:
import os
import sys


ENV_CONFIG = {
    "LANGSMITH_TRACING": {"label": "LangSmith Tracing", "is_flag": True},
    "LANGSMITH_API_KEY": {"label": "LangSmith API Key", "is_secret": True},
    "LANGSMITH_PROJECT": {
        "label": "LangSmith Project",
        "default": "default",
    },
    "GROQ_API_KEY": {"label": "Groq API Key", "is_secret": True},
}


def print_env_status():
    """Validates and prints environment variable status in a structured ASCII format."""
    header = "ENVIRONMENT CONFIGURATION CHECK"
    width = 65

    print("\n" + "=" * width)
    print(f" {header.center(width - 2)} ")
    print("=" * width)
    print(f"{'VARIABLE / SERVICE':<25} | {'STATUS / VALUE':<20} | {'STATE':<10}")
    print("-" * width)

    all_critical_passed = True

    for env_var, meta in ENV_CONFIG.items():
        val = os.getenv(env_var)
        label = meta.get("label", env_var)

        if meta.get("is_secret"):
            if val:
                status_str = "Loaded (Masked)"
                state = "[ PASS ]"
            else:
                status_str = "Missing Key"
                state = "[ FAIL ]"
                all_critical_passed = False

        elif meta.get("is_flag"):
            if val and val.lower() in ("true", "1", "yes"):
                status_str = "Enabled"
                state = "[ ACTIVE ]"
            elif val and val.lower() in ("false", "0", "no"):
                status_str = "Disabled"
                state = "[ OFF ]"
            else:
                status_str = f"Unset ({val if val else 'default'})"
                state = "[ WARN ]"

        else:
            default_val = meta.get("default", "N/A")
            status_str = val if val else f"{default_val} (default)"
            state = "[ INFO ]"

        print(f"{label:<25} | {status_str:<20} | {state:<10}")

    print("=" * width)

    if all_critical_passed:
        print(" SUCCESS: All critical API keys and configurations loaded.")
    else:
        print(" WARNING: Missing required API keys. Services may fail.")

    print("=" * width + "\n")


if __name__ == "__main__":
    print_env_status()


                 ENVIRONMENT CONFIGURATION CHECK                 
VARIABLE / SERVICE        | STATUS / VALUE       | STATE     
-----------------------------------------------------------------
LangSmith Tracing         | Enabled              | [ ACTIVE ]
LangSmith API Key         | Loaded (Masked)      | [ PASS ]  
LangSmith Project         | langsmith-lab        | [ INFO ]  
Groq API Key              | Loaded (Masked)      | [ PASS ]  
 SUCCESS: All critical API keys and configurations loaded.



In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

# LangSmith intercepts this call automatically via the three env vars we set.
response = llm.invoke("What is a LangSmith Run? Answer in 2 sentences.")
print(response.content)

A LangSmith Run is a single, tracked execution of a language‑model operation—such as an LLM call, chain, or agent step—capturing its inputs, outputs, metadata, and timing. These runs are logged to LangSmith’s observability platform so developers can trace, debug, and evaluate the performance of their LLM‑powered applications.


In [4]:
import re
from langsmith import traceable


# Tool: keyword search over the real document 
@traceable(run_type="tool", name="doc_keyword_search")
def search_document(query: str, top_k: int = 3) -> list:
    """Searches llm_production_guide.txt by keyword overlap. Visible as a Tool Run."""
    with open("data/llm_production_guide.txt", encoding="utf-8") as f:
        text = f.read()
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 80]
    keywords   = set(re.findall(r"\b\w{4,}\b", query.lower()))
    ranked     = sorted(paragraphs,
                        key=lambda p: sum(1 for kw in keywords if kw in p.lower()),
                        reverse=True)
    return ranked[:top_k]


In [5]:
# Chain: orchestrates search → LLM → answer 
@traceable(run_type="chain", name="doc_qa_pipeline")
def doc_qa(question: str) -> str:
    """Parent chain. LangSmith shows: doc_qa_pipeline → doc_keyword_search + ChatGroq."""
    sections = search_document(question)            # ← child Tool Run
    context  = "\n\n".join(sections)
    prompt   = f"Context:\n{context}\n\nQuestion: {question}\nAnswer concisely:"
    return llm.invoke(prompt).content               # ← child LLM Run

In [6]:
answer = doc_qa("What is LLM Gateway?")
print(f"Answer: {answer[:300]}...")

Answer: **LLM Gateway** – a unified proxy layer that sits between your app and various LLM providers (e.g., OpenAI, Anthropic, Groq, Gemini). It routes requests, handles automatic fall‑backs, performs semantic caching, balances load across API keys, and centralizes cost and usage tracking, ensuring resilien...


In [7]:
from langsmith import get_current_run_tree 

@traceable(run_type="chain", name="support-query")
def support_qa(question: str, user_id: str, session_id: str) -> str:
    run = get_current_run_tree()
    if run:
        run.metadata.update({
            "user_id":    user_id,
            "session_id": session_id,
            "feature":    "customer-support",
            "env":        "production",
        })
        run.tags = ["production", "support-bot", "groq"]
    return llm.invoke(question).content

In [8]:
list_of_queriers = [
    ("mustafa",   "sess_001", "What is prompt injection and how do we prevent it?"),
    ("mehmet",   "sess_002", "What are best practices for LLM output validation?"),
    ("ali", "sess_003", "How do we monitor LLM costs in production?"),
]

In [9]:
# Run three different users — each trace is tagged for filtering
for user, session, q in list_of_queriers:
    answer = support_qa(q, user_id=user, session_id=session)
    print(f"[{user}] {answer[:150]}...\n")

[mustafa] ## Prompt Injection – A Quick Primer

**Prompt injection** (sometimes called *prompt hijacking* or *jailbreak attack*) is a class of adversarial techn...

[mehmet] ## TL;DR  
Treat the LLM as a **probabilistic generator**, not a trusted oracle.  
Validate its output at **multiple layers**—syntactic, semantic, fac...

[ali] ## Monitoring LLM Costs in Production – A Practical Playbook  

Below is a **step‑by‑step framework** you can adopt (or cherry‑pick) to keep a tight g...



In [10]:
import os
from langsmith import get_current_run_tree
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [11]:
# Load + split the real guide document 
loader   = TextLoader("data/llm_production_guide.txt", encoding="utf-8")
raw_docs = loader.load()
print(f"Loaded: {len(raw_docs[0].page_content):,} characters")

Loaded: 11,679 characters


In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks   = splitter.split_documents(raw_docs)
print(f"Chunks: {len(chunks)}  (avg {sum(len(c.page_content) for c in chunks)//len(chunks)} chars each)")

Chunks: 27  (avg 430 chars each)


In [13]:
print("\nEmbedding chunks via Gemini API (gemini-embedding-2-preview)…")
embeddings  = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅  FAISS index ready")


Embedding chunks via Gemini API (gemini-embedding-2-preview)…
✅  FAISS index ready


In [14]:
# Traced RAG function — @traceable, no LCEL 
@traceable(run_type="chain", name="production_guide_rag")
def rag(question: str, user_id: str = "anonymous") -> str:
    docs    = retriever.invoke(question)
    context = "\n\n".join(f"[chunk {i+1}] {d.page_content}" for i, d in enumerate(docs))
    prompt  = (
        f"Answer based ONLY on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer concisely:"
    )
    run = get_current_run_tree()
    if run:
        run.metadata.update({"user_id": user_id, "chunks_retrieved": len(docs)})
    return llm.invoke(prompt).content

In [15]:
#  Test with two questions 
for q, uid in [
    ("What are the main LLM security risks in production?", "student_01"),
    ("How should we evaluate LLM outputs for quality?",     "student_02"),
]:
    answer = rag(q, user_id=uid)
    print(f"\nQ: {q}")
    print(f"A: {answer[:250]}...")



Q: What are the main LLM security risks in production?
A: The primary production‑time LLM threats highlighted in the provided material are:

1. **LLM06 – Sensitive Information Disclosure**  
   * The model leaks private data from its training set, system prompt, or retrieved documents.  
   * Especially ris...

Q: How should we evaluate LLM outputs for quality?
A: Evaluate LLM outputs with a systematic, reproducible pipeline rather than “vibes.”  
1. **Create a ground‑truth set** of correct answers—ideally by domain experts, or automatically using a stronger model or verifiable closed‑book questions.  
2. **Us...


In [16]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
from langchain_community.utilities import GoogleSerperAPIWrapper 

In [17]:
serper = GoogleSerperAPIWrapper()

In [18]:
@tool
def search_local_docs(query: str) -> str:
    """
    Search the internal LLM production guide.

    IMPORTANT:
    - Use only once per question.
    - After receiving results, answer the user.
    - Do not call repeatedly.
    """

    docs = vectorstore.similarity_search(query, k=3)

    if not docs:
        return "No relevant documents found."

    response = "\n\n".join(
        f"[Chunk {i+1}]\n{doc.page_content[:700]}"
        for i, doc in enumerate(docs)
    )

    # Prevent huge context windows
    return response[:2500]

In [19]:
@tool
def google_search(query: str) -> str:
    """
    Search the web for recent information.

    IMPORTANT:
    - Use only once per question.
    - After receiving results, answer the user.
    - Do not search again unless absolutely required.
    """

    try:
        result = serper.run(query)

        if not result:
            return "No search results found."

        return str(result)[:2500]

    except Exception as e:
        return f"Search failed: {str(e)}"


In [20]:
agent = create_agent(
    model=llm,
    tools = [search_local_docs , google_search],
    system_prompt=""" 
    
    You are a research assistant.

    You have two tools:

    1. search_local_docs
    - Use for RAG, security, evaluation, monitoring,
    prompt engineering, guardrails, deployment.

    2. google_search
    - Use for current events, news,
    regulations, recent AI developments.

    Rules:

    1. Call a tool ONLY if needed.
    2. Never call the same tool more than once.
    3. Maximum TWO total tool calls.
    4. After receiving tool results, provide the final answer.
    5. Do NOT continue searching if enough information exists.
    6. Do NOT loop.
    7. If one tool gives sufficient information,
    answer immediately.
    
    """
)

In [21]:
def run_agent(question: str):

    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        },
        config={
            "recursion_limit": 10
        }
    )

    tools_used = []

    for msg in result["messages"]:
        if isinstance(msg, ToolMessage):
            tools_used.append(msg.name)

    final_answer = ""

    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage):
            final_answer = msg.content
            break

    return final_answer, list(dict.fromkeys(tools_used))


In [22]:
queries = [
    (
        "What are LLM prompt injection attacks and how do we defend against them?",
        "search_local_docs"
    ),
    (
        "What are the latest AI regulations passed in 2025?",
        "google_search"
    ),
    (
        "How does RAG work and what are the latest open-source RAG frameworks in 2025?",
        "both"
    )
]

In [23]:
for question, expected in queries:

    print("\n" + "=" * 80)
    print("QUESTION:")
    print(question)

    print("\nEXPECTED:")
    print(expected)

    answer, tools = run_agent(question)

    print("\nTOOLS USED:")
    print(tools)

    print("\nANSWER:")
    print(answer[:500])

print("\n✅ Completed successfully")


QUESTION:
What are LLM prompt injection attacks and how do we defend against them?

EXPECTED:
search_local_docs

TOOLS USED:
['search_local_docs']

ANSWER:
**What a “prompt‑injection” attack is**

A prompt‑injection attack is when an adversary supplies a user‑generated message (or a piece of retrieved text) that is crafted to **override or tamper with the LLM’s system prompt** – the set of instructions that steer the model’s behavior.  

There are two common flavours:

| Type | How it works | Example |
|------|--------------|---------|
| **Direct injection** | The attacker embeds malicious instructions directly in the user’s input, hoping the model

QUESTION:
What are the latest AI regulations passed in 2025?

EXPECTED:
google_search

TOOLS USED:
['google_search']

ANSWER:
I’m sorry—I wasn’t able to retrieve up‑to‑date information about 2025 AI legislation. Based on the most recent material that’s publicly available (up to 2024), the major regulatory efforts that were expected to culm